In [43]:
from planter.database.utils.duckdb_utils import create_duckdb, merge_duckdbs, update_clusters
from pathlib import Path
import glob

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# Load the current master and examine it

## DatabaseManager API Reference

The `DatabaseManager` provides structured access to the database with pre-built query utilities:

### Core Components

- **`db.con`**: Raw DuckDB connection for custom queries
- **`db.queries`**: QueryManager with specialized query components

### Query Components

#### 1. **`db.queries.sequences`** - Sequence Queries
- `get_by_id(seqhash_id)` - Get complete info for a specific sequence
- `get_expression_summary(sample_id=None)` - Summary of expression data
- `get_expression_distribution(sample_id=None)` - Expression level distribution (low/med/high/very high TPM)
- `get_top_expressed_sequences(limit=10, sample_id=None)` - Top expressed sequences by TPM
- `get_expression_for_sequence(seqhash_id)` - Expression data for a sequence across all samples
- `get_annotation_with_expression(sample_id=None, limit=50)` - Annotations with expression levels
- `search_sequences(sample_ids=None, min_length=None, max_length=None, description=None, organism=None, cog_categories=None, go_terms=None, limit=100)` - Advanced sequence search

#### 2. **`db.queries.clusters`** - Cluster Queries
- `get_cluster_info(cluster_id)` - Detailed info for a specific cluster
- `get_cluster_stats()` - Summary statistics for all clusters

#### 3. **`db.queries.samples`** - Sample Queries
- `get_metadata(sample_id)` - Complete sample information including metadata

#### 4. **`db.queries.organisms`** - Organism Queries
- `get_summary()` - Summary of sequence counts per organism

### Available SQL Queries (in `planter/database/queries/sql/`)
- annotation_with_expression
- cluster_info, cluster_stats
- database_summary
- ec_number_summary
- expression_distribution, expression_summary
- go_term_summary
- organism_summary
- sample_metadata, sample_stats
- search_samples_by_id
- search_sequence_by_seqhash, search_sequences
- sequence_annotations, sequence_by_id, sequence_expression
- top_expressed_sequences


In [8]:
from planter.database.query_manager import DatabaseManager
import duckdb

# Load the master database
MASTER_DB_PATH = "/mnt/data4/master.duckdb"

# Option 1: Using DatabaseManager (provides query utilities)
db = DatabaseManager(MASTER_DB_PATH)
print(f"✓ Loaded master database from {MASTER_DB_PATH}")
print(f"Available query utilities: sequences, clusters, organisms, samples")

# Option 2: Direct DuckDB connection (for custom queries)
# con = duckdb.connect(MASTER_DB_PATH)

# Check tables in the database
tables = db.con.execute("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'main'
    ORDER BY table_name
""").fetchdf()

print(f"\nTables in master database:")
for table in tables['table_name']:
    count = db.con.execute(f"SELECT COUNT(*) as cnt FROM {table}").fetchone()[0]
    print(f"  - {table}: {count:,} rows")


✓ Loaded master database from /mnt/data4/master.duckdb
Available query utilities: sequences, clusters, organisms, samples

Tables in master database:
  - annotations: 2,421,548 rows
  - cluster_members: 2,869,618 rows
  - clusters: 449,541 rows
  - ec_numbers: 679,339 rows
  - expression: 3,444,094 rows
  - gene_protein_map: 1,098,901 rows
  - go_terms: 88,117,109 rows
  - kegg_info: 381,414 rows
  - schema_version: 0 rows
  - sequences: 2,902,248 rows
  - sra_metadata: 115 rows


In [27]:
# Example Usage of DatabaseManager

# 1. Get organism summary
print("=== Organism Summary ===")
organism_df = db.queries.organisms.get_summary()
display(organism_df.head())

# 2. Get cluster statistics
print("\n=== Cluster Statistics ===")
cluster_stats = db.queries.clusters.get_cluster_stats()
display(cluster_stats.head())

# 3. Get expression summary (overall)
print("\n=== Expression Summary ===")
expr_summary = db.queries.sequences.get_expression_summary()
display(expr_summary)

# 4. Get top expressed sequences
print("\n=== Top 5 Expressed Sequences ===")
top_sequences = db.queries.sequences.get_top_expressed_sequences(limit=5)
display(top_sequences)

# 5. Custom query using db.con
print("\n=== Custom Query: Table Row Counts ===")
custom_query = """
SELECT 
    'sequences' as table_name, COUNT(*) as row_count FROM sequences
    UNION ALL
SELECT 'annotations', COUNT(*) FROM annotations
    UNION ALL
SELECT 'clusters', COUNT(*) FROM clusters
    UNION ALL
SELECT 'expression', COUNT(*) FROM expression
ORDER BY row_count DESC
"""
custom_result = db.con.execute(custom_query).fetchdf()
display(custom_result)


=== Organism Summary ===


,organism,sample_count,total_sequences,avg_sequence_length,annotated_sequences,percent_annotated,sequences_in_clusters,bioprojects
0,Silene latifolia subsp. alba,11,300466,370.44,280127,93.23,7168,PRJEB36078
1,Xanthoria parietina,6,289959,329.35,205284,70.80,21771,PRJNA584072; PRJNA584075; PRJNA584074; PRJNA58...
2,Digitalis purpurea,8,260413,344.38,243744,93.60,15373,PRJEB21674; PRJNA80007; PRJNA929980; PRJNA985863
3,Matricaria chamomilla var. recutita,6,240845,359.34,222539,92.40,3502,PRJNA382469
4,Gyalolechia flavorubescens,18,220334,333.85,168318,76.39,15698,PRJNA210248



=== Cluster Statistics ===


,total_clusters,avg_cluster_size,min_cluster_size,max_cluster_size,median_cluster_size
0,205478,1.129328,1,33,1



=== Expression Summary ===


,sample_id,expression_records,min_tpm,max_tpm,mean_tpm,median_tpm,std_tpm,mean_reads,total_reads
0,all,3444094,0.0,176777.79,24.62,5.3,243.36,720.64,2.481963e+09



=== Top 5 Expressed Sequences ===


,gene_seqhash_id,sample_id,original_sample,tpm,num_reads,effective_length,sequence_length,is_representative,description,preferred_name
0,v1_DLS_140ea83780706dc46046b3a3da85cbf0bf1e777...,SRR19619614,SRR19619614,90724.487821,37444.833,387.424,112,False,Belongs to the aldehyde dehydrogenase family,acoD
1,v1_DLS_d6d5dca86de3ac3a807c06bdb4fb7fe2f1cf9d0...,SRR19619613,SRR19619613,85872.156533,576356.971,4660.151,636,False,Function proposed based on presence of conserv...,-
2,v1_DLS_eb825bfdb36bc9e0990a12623d2dde8fde6188c...,SRR19619613,SRR19619613,70923.219297,893908.006,8751.151,415,True,Rieske 2Fe-2S domain protein,-
3,v1_DLS_5e99d825d9f5711603af19c92cae8eabf472093...,SRR19619613,SRR19619613,58399.399395,509803.349,6061.151,736,True,Dienelactone hydrolase family,-
4,v1_DLS_3bc6f20e0ca25ea05eadf04d1cf4b6561602104...,SRR22105605,SRR22105605,50845.689693,460552.291,502.821,168,False,Bulb-type mannose-specific lectin,-



=== Custom Query: Table Row Counts ===


,table_name,row_count
0,expression,3444094
1,sequences,2902248
2,annotations,2421548
3,clusters,205478


# Inspect the new clustering assignments

In [ ]:
cluster_tsv = "/mnt/data4/planter_outputs/repseq_v2/output118/newClusterDB.tsv"
clusters = pd.read_csv(cluster_tsv, header=None, sep='\t')
clusters.shape

# Grab the most/least redunant
clusters[0].value_counts()

0
v1_DLS_9839fd37cab91a5a5c5ecb4025a1a05e0c9c4fa6a1025d5d3d791fc0544f6249.p1     33
v1_DLS_bcc7d7fbb2299e94f1a504944dd66c7cbe58760c082cd70ed04e738f2b6fd33a.p1     29
v1_DLS_78f307f547715fc439b6e1363655d8b9159df196477e48fdba53903c502c1529.p2     26
v1_DLS_dc17346996d5167a0a49cd77dee338a2830f01111be5a1b268e659206fa6f361.p1     25
v1_DLS_90de4e0fabf939327bd48cc0b2165c1a47c975aef9d35424d7804cc75170597c.p1     25
                                                                               ..
v1_DLS_67f3a27efbe5926c96768c566d87d794cf648442e1bf65c7229352c2165b7081.p1      1
v1_DLS_ab15952db45972f2cecced5d6ee061916924465f96c2038c67eb279001f73bb5.p4      1
v1_DLS_34f01773866830bf3667a521511c239c8a2519d9bcd7cd09241f627ad909a0e0.p1      1
v1_DLS_db0c39df9c258416c4c6dd7d06fa4f65493bc6688b2570132d9788a37bb72146.p14     1
v1_DLS_e0364545fbac5a21d2daf011e4763644ddfdf3375fec3632b4c8b20c79436ef9.p1      1
Name: count, Length: 271977, dtype: int64

In [23]:
import pandas as pd
from planter.database.utils.duckdb_utils import update_clusters
from pathlib import Path

master_db = "/mnt/data4/master.duckdb"
# Update the database
update_clusters(
    db_path=master_db,
    tsv_path=cluster_tsv,
    backup_first=True,  # Creates a backup before updating
    handle_duplicates="replace"  # Options: "replace", "ignore", or "error"
)

Starting cluster update for database: /mnt/data4/master.duckdb
Using clustering data from: /mnt/data4/planter_outputs/repseq_v2/output118/newClusterDB.tsv
Duplicate handling strategy: replace
Creating backup at: /mnt/data4/master.duckdb.backup
Beginning database transaction
Loading cluster data from TSV...
Loaded 305702 entries from clustering TSV
Strategy is 'replace': Dropping existing cluster tables...
Recreating cluster tables...
Identifying missing sequences...
Found 66499 unique sequences missing from database
Found 66499 unique representatives missing from database
Examples of missing sequences:
  1. v1_DLS_ac26116ed65ed4fd5fbe70a9f2b33503007acf0248c2490a22e18b48dbc083c8.p2
  2. v1_DLS_41fbef10fdc95786706cd47d99cbe0d7353d26000783da256e81e04e3b96c33c.p1
  3. v1_DLS_69c2973b6a7d2ba42fd5aeffb5e7db8c362f5c57378e8d34f9ea7c1ef050386a.p4
  4. v1_DLS_74b239ea455291fb34f3fbb48dbfcaba28334a129381c42ca7ca3a85c02f3deb.p1
  5. v1_DLS_003f60c19f9825fd7d350ad914ec408000923c6016f0e66d06df880d26

'/mnt/data4/master.duckdb.backup'

In [26]:
# Test 2: Check all Rhodiola rosea sequences and their clustering
query_rhodiola = """
SELECT
    LENGTH(s.sequence) as seq_length,
    s.is_representative,
    s.repseq_id,
    cm.cluster_id,
    c.size as cluster_size,
    a.preferred_name,
    a.description
FROM sequences s
LEFT JOIN sra_metadata sm ON s.sample_id = sm.sample_id
LEFT JOIN annotations a ON s.seqhash_id = a.seqhash_id
LEFT JOIN cluster_members cm ON s.seqhash_id = cm.seqhash_id
LEFT JOIN clusters c ON cm.cluster_id = c.cluster_id
WHERE sm.organism LIKE '%Rhodiola%'
  AND a.description LIKE '%glycosyltransferase%'
ORDER BY seq_length DESC
LIMIT 50
"""

result2 = db.con.execute(query_rhodiola).fetchdf()
print("\nAll Rhodiola UGT-like sequences:")
display(result2)


All Rhodiola UGT-like sequences:


,seq_length,is_representative,repseq_id,cluster_id,cluster_size,preferred_name,description
0,1187,True,v1_DLS_05379cb604537e1572d216ca61b3a85bb2c6adb...,None,NaN,-,Belongs to the glycosyltransferase 2 family
1,1149,False,v1_DLS_74b75ad20454d995b73bfa2649bc39c34e52d15...,None,NaN,-,Belongs to the glycosyltransferase 2 family
2,1141,False,v1_DLS_74b75ad20454d995b73bfa2649bc39c34e52d15...,None,NaN,-,Belongs to the glycosyltransferase 2 family
3,1141,False,v1_DLS_74b75ad20454d995b73bfa2649bc39c34e52d15...,None,NaN,-,Belongs to the glycosyltransferase 2 family
4,1141,False,v1_DLS_74b75ad20454d995b73bfa2649bc39c34e52d15...,None,NaN,-,Belongs to the glycosyltransferase 2 family
5,1085,False,v1_DLS_1838167318a60a52f177c8be0360383c7e3e93c...,None,NaN,-,Belongs to the glycosyltransferase 2 family. P...
6,1085,False,v1_DLS_1838167318a60a52f177c8be0360383c7e3e93c...,None,NaN,-,Belongs to the glycosyltransferase 2 family. P...
7,1085,False,v1_DLS_05379cb604537e1572d216ca61b3a85bb2c6adb...,None,NaN,-,Belongs to the glycosyltransferase 2 family. P...
8,1084,False,v1_DLS_8916d9621af74dba4fe17130361f8f9824e82c3...,None,NaN,CESA1,Belongs to the glycosyltransferase 2 family. P...
9,1084,False,v1_DLS_8916d9621af74dba4fe17130361f8f9824e82c3...,None,NaN,CESA1,Belongs to the glycosyltransferase 2 family. P...
